# Module 10 — Python integration

End-to-end tour: psycopg 3 (sync), asyncpg, SQLAlchemy 2.x (sync + async), Alembic.

Run cells top-to-bottom. `seed/schema.sql` and `seed/data.sql` must already be loaded.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
DSN = f"postgresql://{os.environ['PGUSER']}:{os.environ['PGPASSWORD']}@{os.environ['PGHOST']}:{os.environ['PGPORT']}/{os.environ['PGDATABASE']}"
DSN_SQLA       = os.environ['DATABASE_URL']
DSN_SQLA_ASYNC = os.environ['DATABASE_URL_ASYNC']
print('OK')

## 1. psycopg 3 — sync

Parameterised queries, dict rows, explicit transactions.

In [ ]:
import psycopg
from psycopg.rows import dict_row

with psycopg.connect(DSN, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute('SELECT id, email FROM app.users WHERE id = %s', (1,))
        print(cur.fetchone())
        cur.execute(
            """SELECT id, title FROM app.posts WHERE tags @> %s ORDER BY id LIMIT 5""",
            (['pgvector'],),
        )
        for r in cur:
            print(r)

### Bulk load with COPY

Fastest way to ingest data.

In [ ]:
with psycopg.connect(DSN) as conn:
    with conn.cursor() as cur:
        cur.execute('CREATE TEMP TABLE demo_load (i int, v text)')
        with cur.copy('COPY demo_load (i, v) FROM STDIN') as cp:
            for i in range(10_000):
                cp.write_row((i, f'row-{i}'))
        cur.execute('SELECT count(*) FROM demo_load')
        print('rows:', cur.fetchone()[0])

### Streaming with a server-side cursor

When the result set is too big to fit in memory.

In [ ]:
with psycopg.connect(DSN) as conn:
    with conn.cursor(name='stream_demo') as cur:
        cur.itersize = 1000
        cur.execute('SELECT id FROM app.posts')
        total = sum(1 for _ in cur)
        print('streamed rows:', total)

## 2. asyncpg

High-throughput async driver. Note `$1` placeholders.

In [ ]:
import asyncpg, asyncio

async def demo():
    pool = await asyncpg.create_pool(DSN, min_size=2, max_size=5)
    async with pool.acquire() as conn:
        rows = await conn.fetch(
            'SELECT id, title FROM app.posts WHERE author_id = $1 ORDER BY id LIMIT $2',
            7, 3,
        )
        for r in rows:
            print(dict(r))
    await pool.close()

await demo()

## 3. SQLAlchemy 2.x — Core

In [ ]:
from sqlalchemy import create_engine, MetaData, Table, select, text

engine = create_engine(DSN_SQLA, pool_size=5, max_overflow=10)
md = MetaData(schema='app')
users = Table('users', md, autoload_with=engine)
posts = Table('posts', md, autoload_with=engine)

with engine.connect() as conn:
    stmt = (
        select(users.c.email, posts.c.title)
        .join_from(users, posts, users.c.id == posts.c.author_id)
        .where(posts.c.published_at.is_not(None))
        .limit(5)
    )
    for row in conn.execute(stmt):
        print(row)

## 4. SQLAlchemy 2.x — ORM

Declarative models, typed mappings, sessions.

In [ ]:
from datetime import datetime
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session
from sqlalchemy import BigInteger, Text, DateTime, select as sa_select

class Base(DeclarativeBase):
    pass

class User(Base):
    __tablename__ = 'users'
    __table_args__ = {'schema': 'app'}
    id: Mapped[int] = mapped_column(BigInteger, primary_key=True)
    email: Mapped[str] = mapped_column(Text)
    full_name: Mapped[str] = mapped_column(Text)
    created_at: Mapped[datetime] = mapped_column(DateTime(timezone=True))

with Session(engine) as session:
    user = session.scalars(sa_select(User).where(User.id == 1)).one()
    print(user.id, user.email, user.full_name)

## 5. SQLAlchemy async over asyncpg

In [ ]:
from sqlalchemy.ext.asyncio import create_async_engine, AsyncSession

async_engine = create_async_engine(DSN_SQLA_ASYNC, pool_size=5)

async def demo_async():
    async with AsyncSession(async_engine) as session:
        result = await session.scalars(sa_select(User).where(User.id <= 5))
        for u in result:
            print(u.id, u.email)
    await async_engine.dispose()

await demo_async()

## 6. Alembic — minimal recipe

In a real project run these in a shell at the repo root:

```powershell
alembic init migrations
# edit migrations/env.py: set sqlalchemy.url = DATABASE_URL, target_metadata = Base.metadata
alembic revision --autogenerate -m 'add user.bio'
alembic upgrade head
alembic downgrade -1
```

The generated `versions/*.py` contains `upgrade()` and `downgrade()` you must review before applying to a real database. Alembic occasionally misses CHECK constraints, partial indexes, and generated columns.